# Milestone 4: Final Model Pipeline



## Step 1: Persistence model

**Goal:** Compare naive persistence forecasts built from **48-hour** and **1-week (168-hour)** price lags on the same held-out test window. We report **MAE** and **RMSE** so that we can judge which horizon is more predictive and thus useful for our final LSTM model.

**Definitions:** For hourly data, persistence means $\hat{y}_t = y_{t-k}$ using the lag-$k$ column in features. One week at the same hour is $k=168$ (7×24).

## Setup

Imports and a fixed random seed.

In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

RNG_SEED = 42
np.random.seed(RNG_SEED)

## Load data and add long lags

Uses initial_data.csv. That file already has short lags; we add **`price_lag_48`** and **`price_lag_168`** from `price`, then drop rows with missing lags so metrics are only on valid rows.

In [3]:
DATA_PATH = "initial_data.csv"

df = pd.read_csv(DATA_PATH)
df["datetime_beginning_ept"] = pd.to_datetime(df["datetime_beginning_ept"])
df = df.sort_values("datetime_beginning_ept").set_index("datetime_beginning_ept")

df["price_lag_48"] = df["price"].shift(48)
df["price_lag_168"] = df["price"].shift(7 * 24)

lag_cols = ["price_lag_48", "price_lag_168"]
df = df.dropna(subset=lag_cols + ["price"])

print("Shape after lag dropna:", df.shape)
print("Index range:", df.index.min(), "→", df.index.max())
df[lag_cols].head()

Shape after lag dropna: (8568, 29)
Index range: 2025-04-15 00:00:00 → 2026-04-07 00:00:00


,price_lag_48,price_lag_168
datetime_beginning_ept,,
2025-04-15 00:00:00,61.521585,48.900998
2025-04-15 01:00:00,44.033375,46.402496
2025-04-15 02:00:00,35.028002,45.297246
2025-04-15 03:00:00,34.611806,46.532548
2025-04-15 04:00:00,35.527041,48.516346


## Time-aware train / validation / test split

Same logic as the baseline notebook: **no shuffling**, chronological **70% / 15% / 15%** so the test period is strictly later in time.

In [6]:
def time_series_split(
    frame: pd.DataFrame,
    target_col: str = "price",
    train_frac: float = 0.70,
    val_frac: float = 0.15,
):
    # Split time-ordered frame into train / val / test (no shuffle).
    n = len(frame)
    train_end = int(n * train_frac)
    val_end = int(n * (train_frac + val_frac))

    feature_cols = [c for c in frame.columns if c != target_col]

    train_df = frame.iloc[:train_end]
    val_df = frame.iloc[train_end:val_end]
    test_df = frame.iloc[val_end:]

    X_train = train_df[feature_cols].copy()
    y_train = train_df[target_col].copy()
    X_val = val_df[feature_cols].copy()
    y_val = val_df[target_col].copy()
    X_test = test_df[feature_cols].copy()
    y_test = test_df[target_col].copy()

    return X_train, y_train, X_val, y_val, X_test, y_test


X_train, y_train, X_val, y_val, X_test, y_test = time_series_split(df)

print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)
print("Test window:", X_test.index.min(), "→", X_test.index.max())

Train: (5997, 28) | Val: (1285, 28) | Test: (1286, 28)
Test window: 2026-02-12 10:00:00 → 2026-04-07 00:00:00


## Persistence models and metrics

**Persistence:** $\hat{y}_t$ equals the price observed $k$ hours earlier (`price_lag_k`). **MAE** is typical error; **RMSE** weights spikes more, matching our dual-metric rationale from MS3.

In [7]:
def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return mae, rmse


rows = []
for lag_name, col in [
    ("Persistence (48h)", "price_lag_48"),
    ("Persistence (1 week)", "price_lag_168"),
]:
    y_pred = X_test[col].to_numpy()
    mae, rmse = regression_metrics(y_test, y_pred)
    rows.append({"Model": lag_name, "Lag hours": int(col.rsplit("_", 1)[-1]), "MAE": mae, "RMSE": rmse})

results = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
results

,Model,Lag hours,MAE,RMSE
0,Persistence (48h),48,14.175086,22.875045
1,Persistence (1 week),168,23.161591,48.136436


We see that the persistence model for 48 hour lag does much better in both MAE and RMSE compared to the 1 week persistence model. Based on this, we will use the 48 hour lag price as a feature in our final LSTM model.